# Autonomous Trailer

**ICS 3202 — Practical Exercise 4**  
Arnold Richard Mwendwa - 172340

Abdullahi Issa Mohamed - 157327

Mohamedek Yusuf Aden - 190134

## Step 1: Identify the task

We modelled a self-driving truck/trailer planning a journey from Nairobi CBD to Kisumu Town. We represented towns as states, roads as directed edges, and each edge's driving distance in kilometres. We set the planning goal as finding a route with the smallest total distance in the supplied graph and identifying the next town on that route. We recognized that the real driving environment was partially observable and dynamic, so our graph did not tell us whether roads were currently safe. Current road, traffic, and weather information would also be needed before driving. Our shortest mapped route was therefore a planning suggestion, not a guarantee of safety or travel time.


## Step 2: Assemble relevant knowledge

We used the Practical 3 graph, which contained 20 towns and 52 directed road-distance facts measured in kilometres using OSRM/OpenStreetMap data. Our search-tree notebook used Nairobi CBD as the start and Kisumu Town as the goal and demonstrated bounded depth-first search. We used the full weighted graph for route planning because the bounded search tree illustrated exploration but did not prove which route was shortest.

We copied the distances below from the supplied graph notebook outputs, so our example ran without making another API call. These were map facts, not live traffic, road-closure, weather, or safety observations. If we updated the map, we would replace them with the latest graph export and record the source.


## Step 3: Define the vocabulary

We used these terms in our knowledge base:

- `Town(name)`: a location/state in the graph.
- `Road(from_town, to_town)`: a directed road connection.
- `Distance(from_town, to_town, km)`: the measured driving distance on that directed edge.
- `At(town)` and `Destination(town)`: the current trip facts.
- `Blocked(from_town, to_town)`: an optional verified, current observation that ruled out a mapped edge.
- `Route(path, total_km)`: a path and its summed distance.

Our map did not contain live road-condition facts. The absence of a blocked-road fact meant “not reported blocked,” not “verified clear.”


## Step 4: Encode general knowledge about the domain

We separated general routing rules from the trip's specific facts:

1. We travelled only along directed roads recorded in the graph.
2. We calculated route cost by summing edge distances in kilometres.
3. We excluded an edge when we had a verified blocked-road observation for it.
4. We chose a route with minimum total distance among the remaining paths.
5. When the start or goal was unknown, or no route was found, we reported that instead of inventing a path.

Our knowledge base optimized mapped distance only; it did not prove that a route was currently safe or fastest.


In [1]:
# We encode the 20 towns and 52 directed distances from our Practical 3 graph.
towns = ['Ahero', 'Eldoret', 'Gilgil', 'Homa Bay', 'Kakamega', 'Kericho', 'Kiambu', 'Kisii', 'Kisumu Town', 'Limuru', "Murang'a", 'Nairobi CBD', 'Naivasha', 'Nakuru', 'Nanyuki', 'Nyahururu', 'Nyeri', 'Ruiru', 'Sagana', 'Thika']

road_facts = [
    ('Nairobi CBD', 'Ruiru', 24.78),
    ('Ruiru', 'Nairobi CBD', 25.83),
    ('Ruiru', 'Thika', 26.27),
    ('Thika', 'Ruiru', 24.73),
    ('Nairobi CBD', 'Kiambu', 40.32),
    ('Kiambu', 'Nairobi CBD', 40.91),
    ('Kiambu', 'Limuru', 60.85),
    ('Limuru', 'Kiambu', 64.25),
    ('Nairobi CBD', 'Limuru', 32.09),
    ('Limuru', 'Nairobi CBD', 31.17),
    ('Thika', "Murang'a", 46.65),
    ("Murang'a", 'Thika', 44.25),
    ("Murang'a", 'Sagana', 46.59),
    ('Sagana', "Murang'a", 46.59),
    ('Sagana', 'Nyeri', 68.81),
    ('Nyeri', 'Sagana', 68.71),
    ('Nyeri', 'Nanyuki', 54.35),
    ('Nanyuki', 'Nyeri', 54.51),
    ('Nanyuki', 'Nyahururu', 112.19),
    ('Nyahururu', 'Nanyuki', 112.02),
    ('Nyeri', 'Nyahururu', 90.22),
    ('Nyahururu', 'Nyeri', 90.20),
    ('Limuru', 'Naivasha', 60.50),
    ('Naivasha', 'Limuru', 57.32),
    ('Nairobi CBD', 'Naivasha', 88.45),
    ('Naivasha', 'Nairobi CBD', 88.49),
    ('Naivasha', 'Gilgil', 48.18),
    ('Gilgil', 'Naivasha', 48.39),
    ('Gilgil', 'Nakuru', 43.78),
    ('Nakuru', 'Gilgil', 43.12),
    ('Nyahururu', 'Nakuru', 63.46),
    ('Nakuru', 'Nyahururu', 62.75),
    ('Nakuru', 'Eldoret', 152.22),
    ('Eldoret', 'Nakuru', 154.97),
    ('Nakuru', 'Kericho', 116.38),
    ('Kericho', 'Nakuru', 116.58),
    ('Eldoret', 'Kakamega', 74.59),
    ('Kakamega', 'Eldoret', 75.08),
    ('Kakamega', 'Kisumu Town', 87.05),
    ('Kisumu Town', 'Kakamega', 87.04),
    ('Kericho', 'Kisumu Town', 72.92),
    ('Kisumu Town', 'Kericho', 72.98),
    ('Kericho', 'Kisii', 113.88),
    ('Kisii', 'Kericho', 113.88),
    ('Kisii', 'Homa Bay', 72.23),
    ('Homa Bay', 'Kisii', 72.23),
    ('Homa Bay', 'Ahero', 106.54),
    ('Ahero', 'Homa Bay', 106.79),
    ('Ahero', 'Kisumu Town', 22.44),
    ('Kisumu Town', 'Ahero', 22.50),
    ('Kisii', 'Ahero', 99.13),
    ('Ahero', 'Kisii', 99.32),
]

knowledge_base = {
    "towns": set(towns),
    "roads": {(start, end): {"distance_km": km} for start, end, km in road_facts},
    "blocked_roads": set(),  # Practical 3 did not provide live closure facts.
    "rules": [
        "Use recorded directed roads only.",
        "Sum edge distances to calculate route cost.",
        "Exclude roads with verified blocked-road facts.",
        "Return a minimum-distance known route or report no route.",
    ],
}

print(f"Knowledge base: {len(knowledge_base['towns'])} towns, "
      f"{len(knowledge_base['roads'])} directed roads")
for rule in knowledge_base["rules"]:
    print("-", rule)


Knowledge base: 20 towns, 52 directed roads
- Use recorded directed roads only.
- Sum edge distances to calculate route cost.
- Exclude roads with verified blocked-road facts.
- Return a minimum-distance known route or report no route.


## Step 5: Add facts to the knowledge base

We implemented functions to add directed distance facts after checking both town names and the distance. We treated each reverse direction as a separate fact because its measured distance could differ. We also added a way to record a verified blocked-road observation without deleting the underlying map connection.


In [2]:
# We add or update directed road facts only after checking their source.
def add_road_fact(kb, start, end, distance_km):
    if start not in kb["towns"] or end not in kb["towns"]:
        raise ValueError("Both endpoints must be towns in the graph.")
    if start == end or not isinstance(distance_km, (int, float)) or distance_km <= 0:
        raise ValueError("Use different endpoints and a positive distance.")
    kb["roads"][(start, end)] = {"distance_km": float(distance_km)}
    return f"Stored {start} -> {end}: {distance_km:g} km"


def mark_road_blocked(kb, start, end):
    if (start, end) not in kb["roads"]:
        raise ValueError("That directed road is not in the map.")
    kb["blocked_roads"].add((start, end))
    return f"Recorded {start} -> {end} as blocked for this trip."


def show_knowledge_base(kb):
    print(f"Towns: {len(kb['towns'])}; directed roads: {len(kb['roads'])}")
    for (start, end), fact in sorted(kb["roads"].items()):
        tag = " [BLOCKED]" if (start, end) in kb["blocked_roads"] else ""
        print(f"{start} -> {end}: {fact['distance_km']:.2f} km{tag}")

print(add_road_fact(knowledge_base, "Ahero", "Kisumu Town", 22.44))


Stored Ahero -> Kisumu Town: 22.44 km


## Steps 6 and 7: Encode a problem instance, pose queries, and get answers

We encoded a sample trip from Nairobi CBD to Kisumu Town, matching Practical 3. We used Dijkstra's algorithm because all recorded edge distances were non-negative. Our answer included the path, total distance, and next town. It was based on the supplied graph and was not live driving authorization.


In [3]:
# We enter the current trip facts and query for a minimum-distance route.
import heapq

current_location = "Nairobi CBD"
destination = "Kisumu Town"


def shortest_route(kb, start, goal):
    """Find a minimum-distance path on mapped, not-known-blocked edges."""
    if start not in kb["towns"] or goal not in kb["towns"]:
        return None
    frontier = [(0.0, start, [start])]
    best = {start: 0.0}
    while frontier:
        cost, town, path = heapq.heappop(frontier)
        if cost > best.get(town, float("inf")):
            continue
        if town == goal:
            return {"path": path, "distance_km": cost}
        for (a, b), fact in kb["roads"].items():
            if a != town or (a, b) in kb["blocked_roads"]:
                continue
            new_cost = cost + fact["distance_km"]
            if new_cost < best.get(b, float("inf")):
                best[b] = new_cost
                heapq.heappush(frontier, (new_cost, b, path + [b]))
    return None


def answer_route_query(kb, start, goal):
    result = shortest_route(kb, start, goal)
    if result is None:
        return {"decision": "no_known_route", "message": "Check the map, town names, and blocked-road facts."}
    if len(result["path"]) == 1:
        return {"decision": "already_at_destination", **result}
    return {
        "decision": "route_found",
        "next_town": result["path"][1],
        **result,
        "qualification": "Mapped-distance result; verify current road and traffic conditions.",
    }

print(answer_route_query(knowledge_base, current_location, destination))


{'decision': 'route_found', 'next_town': 'Naivasha', 'path': ['Nairobi CBD', 'Naivasha', 'Gilgil', 'Nakuru', 'Kericho', 'Kisumu Town'], 'distance_km': 369.71, 'qualification': 'Mapped-distance result; verify current road and traffic conditions.'}


## Step 8: Query the knowledge base

We queried the main journey, a short journey, a trip whose start already equalled its destination, and an unknown town. We compared the answers with the graph facts.


In [4]:
queries = [
    ("Nairobi CBD", "Kisumu Town"),
    ("Ahero", "Kisumu Town"),
    ("Kisumu Town", "Kisumu Town"),
    ("Nairobi CBD", "Unknown Town"),
]
for start, goal in queries:
    print(f"\nQuery: {start} -> {goal}")
    print(answer_route_query(knowledge_base, start, goal))



Query: Nairobi CBD -> Kisumu Town
{'decision': 'route_found', 'next_town': 'Naivasha', 'path': ['Nairobi CBD', 'Naivasha', 'Gilgil', 'Nakuru', 'Kericho', 'Kisumu Town'], 'distance_km': 369.71, 'qualification': 'Mapped-distance result; verify current road and traffic conditions.'}

Query: Ahero -> Kisumu Town
{'decision': 'route_found', 'next_town': 'Kisumu Town', 'path': ['Ahero', 'Kisumu Town'], 'distance_km': 22.44, 'qualification': 'Mapped-distance result; verify current road and traffic conditions.'}

Query: Kisumu Town -> Kisumu Town
{'decision': 'already_at_destination', 'path': ['Kisumu Town'], 'distance_km': 0.0}

Query: Nairobi CBD -> Unknown Town
{'decision': 'no_known_route', 'message': 'Check the map, town names, and blocked-road facts.'}


## Step 9: Debug the knowledge base

We inspected ordinary and edge cases. We temporarily blocked one mapped edge and queried for a route again to check whether our search found an alternative. We kept reverse roads as separate facts with their own measured distances.


In [5]:
debug_cases = [
    ("Nairobi CBD", "Kisumu Town", "main Practical 3 journey"),
    ("Nairobi CBD", "Nairobi CBD", "already at destination"),
    ("Kisumu Town", "Nairobi CBD", "reverse journey"),
    ("Nairobi CBD", "Not a Town", "unknown goal"),
]
print("--- Debugging route queries ---")
for start, goal, note in debug_cases:
    print(note, "=>", answer_route_query(knowledge_base, start, goal))

edge_to_block = ("Nairobi CBD", "Naivasha")
knowledge_base["blocked_roads"].add(edge_to_block)
print("\nAfter blocking Nairobi CBD -> Naivasha:")
print(answer_route_query(knowledge_base, "Nairobi CBD", "Kisumu Town"))
knowledge_base["blocked_roads"].remove(edge_to_block)

invalid_facts = [
    (edge, fact) for edge, fact in knowledge_base["roads"].items()
    if fact["distance_km"] <= 0
    or edge[0] not in knowledge_base["towns"]
    or edge[1] not in knowledge_base["towns"]
]
print("Invalid road facts:", invalid_facts)


--- Debugging route queries ---
main Practical 3 journey => {'decision': 'route_found', 'next_town': 'Naivasha', 'path': ['Nairobi CBD', 'Naivasha', 'Gilgil', 'Nakuru', 'Kericho', 'Kisumu Town'], 'distance_km': 369.71, 'qualification': 'Mapped-distance result; verify current road and traffic conditions.'}
already at destination => {'decision': 'already_at_destination', 'path': ['Nairobi CBD'], 'distance_km': 0.0}
reverse journey => {'decision': 'route_found', 'next_town': 'Kericho', 'path': ['Kisumu Town', 'Kericho', 'Nakuru', 'Gilgil', 'Naivasha', 'Nairobi CBD'], 'distance_km': 369.56, 'qualification': 'Mapped-distance result; verify current road and traffic conditions.'}
unknown goal => {'decision': 'no_known_route', 'message': 'Check the map, town names, and blocked-road facts.'}

After blocking Nairobi CBD -> Naivasha:
{'decision': 'route_found', 'next_town': 'Limuru', 'path': ['Nairobi CBD', 'Limuru', 'Naivasha', 'Gilgil', 'Nakuru', 'Kericho', 'Kisumu Town'], 'distance_km': 373.85

## Step 10: Final reflection

We discussed what we found difficult when abstracting Kenya's roads into a directed graph, how direction-specific distances affected route cost, what the bounded search tree illustrated, and what we would add next (such as verified road closures, traffic, and weather observations). We recognized that minimizing mapped kilometres alone did not ensure safety, fastest travel, or on-time delivery.

**Our reflection:** We spent time collecting and checking road-distance data, and simplifying the network meant leaving out junctions and local road details. Our graph also showed distance rather than live traffic or safety, so the shortest route was not necessarily the fastest or safest.

## Groupwork evidence

![Group Photo](group.jpg)